# Press Start: A Comparative Machine Learning Approach to Video Game Sales Prediction
**COMP 3608 — Machine Learning | B-Rank Mission**

| Member | Student ID |
|--------|-----------|
| Matthew Singh | 816035076 |
| Vinayak Maharaj | 816036176 |
| Jonathan La Borde | 816041435 |

---

## Table of Contents
1. [Environment Setup](#setup)
2. [Problem Statement](#problem)
3. [Dataset Download](#download)
4. [Data Inspection](#inspect)
5. [Preprocessing](#preprocessing)
6. [Exploratory Data Analysis](#eda)
7. [Experimental Design](#design)
8. [Model Training & Evaluation](#models)
9. [Sensitivity Analysis](#sensitivity)
10. [Results Summary](#results)
11. [Discussion & Conclusion](#discussion)


## 1. Environment Setup <a name='setup'></a>

In [ ]:
# Mount Google Drive — data and plots persist across sessions
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR  = '/content/drive/MyDrive/press_start'
DATA_DIR  = os.path.join(BASE_DIR, 'data')
CLEAN_DIR = os.path.join(BASE_DIR, 'clean')
PLOT_DIR  = os.path.join(BASE_DIR, 'plots')

for d in [DATA_DIR, CLEAN_DIR, PLOT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted and directories ready ✓")

In [ ]:
import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot    as plt
import matplotlib.ticker    as mticker
import seaborn              as sns

from sklearn.linear_model    import LinearRegression
from sklearn.tree            import DecisionTreeRegressor
from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics         import mean_squared_error, r2_score
from sklearn.preprocessing   import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

# ── Constants ─────────────────────────────────────────────────────────────────
TARGET_COL   = 'Global_Sales'
FEATURE_COLS = ['Platform', 'Genre', 'Publisher',
                'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Year']
RANDOM_STATE = 42
TEST_SIZE    = 0.2
COLORS       = ['#4C72B0', '#DD8452', '#55A868']

print("Libraries loaded ✓")

## 2. Problem Statement <a name='problem'></a>

The video game industry generates billions of dollars annually, yet publishers and developers frequently struggle to predict how well a game will sell across different platforms and regions. Poor sales forecasting leads to overproduction, misallocated marketing budgets, and financial losses.

**Problem:** Can we accurately predict the global sales figures of a video game given features such as genre, platform, publisher, and regional performance?

**Problem Type:** Regression — predicting a continuous numerical value (global sales in millions of units).

### Stakeholders

| Stakeholder | Need |
|-------------|------|
| Game publishers | Decide how much to invest in a title before release |
| Retail distributors | Manage stock levels and avoid overproduction |
| Independent developers | Identify which genres and platforms offer the best opportunity |

### Why it Matters
A reliable prediction model could save publishers millions in wasted resources and help independent developers make smarter, data-driven decisions about where and how to release their games.


## 3. Dataset Download <a name='download'></a>

All three datasets are sourced from Kaggle. To download them you will need a Kaggle API token (`kaggle.json`).  
Get it at: **kaggle.com → Account → API → Create New Token**


In [ ]:
from google.colab import files

# Upload your kaggle.json when the file picker appears
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install kaggle -q

print("Kaggle API configured ✓")

In [ ]:
datasets_to_download = {
    'dataset1': 'bhushandardivekar/video-game-sales-and-industry-data-1980-2024',
    'dataset2': 'lamskdna/video-games-sales',
    'dataset3': 'thedevastator/global-video-game-sales',
}

for key, slug in datasets_to_download.items():
    out = os.path.join(DATA_DIR, key)
    os.makedirs(out, exist_ok=True)
    print(f"Downloading {slug}...")
    !kaggle datasets download -d {slug} -p {out} --unzip -q
    files_found = os.listdir(out)
    print(f"  → {files_found}\n")

In [ ]:
# ── After running the cell above, update these paths ─────────────────────────
# Look at the output above and set each path to the actual .csv filename found

RAW_PATH_1 = os.path.join(DATA_DIR, 'dataset1', 'vgsales.csv')   # update if needed
RAW_PATH_2 = os.path.join(DATA_DIR, 'dataset2', 'vgsales.csv')   # update if needed
RAW_PATH_3 = os.path.join(DATA_DIR, 'dataset3', 'vgsales.csv')   # update if needed

raw_paths = {'Dataset 1': RAW_PATH_1,
             'Dataset 2': RAW_PATH_2,
             'Dataset 3': RAW_PATH_3}

for name, path in raw_paths.items():
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1024 if exists else 0
    print(f"  {name}: {'✓' if exists else '✗ NOT FOUND'}  ({size:.1f} KB)  {path}")

## 4. Data Inspection <a name='inspect'></a>

Before any cleaning, we inspect the raw datasets to understand their structure, column names, and missing value patterns.


In [ ]:
raw_dfs = {}

for name, path in raw_paths.items():
    df = pd.read_csv(path)
    raw_dfs[name] = df
    print(f"\n{'='*55}")
    print(f"{name}  |  Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    display(df.head(3))
    print("\nMissing values:")
    missing = df.isnull().sum()
    print(missing[missing > 0].to_string() if missing.any() else "  None")

## 5. Preprocessing <a name='preprocessing'></a>

Each dataset uses different column names and structures. We standardise them all to a consistent schema, handle missing values, encode categorical variables, clip extreme outliers, and apply a log₁⁺ transform to the target.

### Why log₁⁺ transform?
Video game sales are heavily right-skewed — a handful of titles (e.g. Wii Sports) sell tens of millions while the vast majority sell under 1M units. Compressing the target with `log1p` brings the distribution closer to normal, which significantly improves model performance.


In [ ]:
def encode_categoricals(df, cols):
    le = LabelEncoder()
    for col in cols:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown').astype(str)
            df[col] = le.fit_transform(df[col])
    return df

def clip_outliers(df, col, upper_q=0.99):
    cap = df[col].quantile(upper_q)
    n   = (df[col] > cap).sum()
    df[col] = df[col].clip(upper=cap)
    print(f"  Clipped {n} outlier(s) in '{col}' at {cap:.2f}M units")
    return df

def clean_dataset(df, name, rename_map=None, log=True):
    print(f"\nCleaning {name}...")
    if rename_map:
        df = df.rename(columns=rename_map)

    # Keep only standard columns that exist
    keep = [c for c in FEATURE_COLS + [TARGET_COL] if c in df.columns]
    df   = df[keep].copy()

    # Drop rows with no target value
    before = len(df)
    df = df.dropna(subset=[TARGET_COL])
    print(f"  Dropped {before - len(df)} rows missing target")

    # Numeric year, fill missing with 0
    if 'Year' in df.columns:
        df['Year'] = pd.to_numeric(df['Year'], errors='coerce').fillna(0).astype(int)

    # Fill missing regional sales with 0
    for col in ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df = encode_categoricals(df, ['Platform', 'Genre', 'Publisher'])
    df = clip_outliers(df, TARGET_COL)

    if log:
        df[TARGET_COL + '_raw'] = df[TARGET_COL]
        df[TARGET_COL]          = np.log1p(df[TARGET_COL])
        print(f"  Applied log1p transform to {TARGET_COL}")

    print(f"  Final shape: {df.shape}")
    return df

In [ ]:
# ── If any column names differ from our standard names, add them here ─────────
# e.g. rename_map_1 = {'global_sales': 'Global_Sales', 'genre': 'Genre'}
rename_map_1 = {}
rename_map_2 = {}
rename_map_3 = {}

df1 = clean_dataset(raw_dfs['Dataset 1'].copy(), 'Dataset 1', rename_map_1)
df2 = clean_dataset(raw_dfs['Dataset 2'].copy(), 'Dataset 2', rename_map_2)
df3 = clean_dataset(raw_dfs['Dataset 3'].copy(), 'Dataset 3', rename_map_3)

datasets = {'Dataset 1': df1, 'Dataset 2': df2, 'Dataset 3': df3}

# Save cleaned CSVs so we never need to rerun this
df1.to_csv(os.path.join(CLEAN_DIR, 'dataset1_clean.csv'), index=False)
df2.to_csv(os.path.join(CLEAN_DIR, 'dataset2_clean.csv'), index=False)
df3.to_csv(os.path.join(CLEAN_DIR, 'dataset3_clean.csv'), index=False)
print("\nCleaned datasets saved to Drive ✓")

In [ ]:
# Quick sanity check — display head of each cleaned dataset
for name, df in datasets.items():
    print(f"\n{name}  |  {df.shape}")
    display(df.head(3))

## 6. Exploratory Data Analysis <a name='eda'></a>

We analyse all three cleaned datasets to understand distributions, feature correlations, and sales trends before model training.


### 6.1 Summary Statistics

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    display(df.describe().round(3))

### 6.2 Target Distribution — Raw vs log₁⁺

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Global Sales Distribution — Raw vs log₁⁺ Transformed',
             fontsize=14, fontweight='bold', y=1.01)

for col_i, (name, df) in enumerate(datasets.items()):
    raw = df[TARGET_COL + '_raw']
    log = df[TARGET_COL]

    for row_i, (data, label, color) in enumerate([
        (raw, 'Raw Sales (millions)', COLORS[col_i]),
        (log, 'log₁⁺(Sales)',         COLORS[col_i]),
    ]):
        ax = axes[row_i, col_i]
        ax.hist(data, bins=60, color=color, edgecolor='white', alpha=0.85)
        ax.set_title(f'{name}\n{label}', fontweight='bold')
        ax.set_xlabel(label)
        ax.set_ylabel('Count')
        ax.text(0.96, 0.95, f'Skew: {data.skew():.2f}',
                transform=ax.transAxes, ha='right', va='top', fontsize=9,
                bbox=dict(boxstyle='round', fc='white', alpha=0.85))

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'target_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Correlation Heatmaps

In [ ]:
for name, df in datasets.items():
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if '_raw' not in c]
    corr = df[num_cols].corr()

    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, ax=ax, linewidths=0.4, annot_kws={'size': 9})
    ax.set_title(f'{name} — Correlation Matrix', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f'correlation_{name.replace(" ","_")}.png'), dpi=150)
    plt.show()

### 6.4 Sales by Genre and Platform
> Note: Genre/Platform are label-encoded integers at this stage. See raw CSVs for readable category names.

In [ ]:
for name, df in datasets.items():
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'{name} — Median log₁⁺ Sales by Category', fontweight='bold')

    for ax, col in zip(axes, ['Genre', 'Platform']):
        if col not in df.columns:
            ax.set_visible(False); continue
        grp = (df.groupby(col)[TARGET_COL]
                 .median()
                 .sort_values(ascending=False)
                 .head(10))
        grp.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='white', width=0.7)
        ax.set_title(f'Top 10 {col} codes by Median Sales')
        ax.set_xlabel(f'{col} (encoded)')
        ax.set_ylabel('Median log₁⁺ Sales')
        ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f'by_category_{name.replace(" ","_")}.png'), dpi=150)
    plt.show()

### 6.5 Sales Over Time

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Total Global Sales by Year (log₁⁺ sum)', fontweight='bold')

for ax, (name, df), color in zip(axes, datasets.items(), COLORS):
    if 'Year' not in df.columns: continue
    yearly = df[df['Year'] > 1980].groupby('Year')[TARGET_COL].sum()
    ax.plot(yearly.index, yearly.values, marker='o', linewidth=2,
            markersize=4, color=color)
    ax.fill_between(yearly.index, yearly.values, alpha=0.15, color=color)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Total log₁⁺ Sales')
    ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'sales_over_time.png'), dpi=150)
plt.show()

### 6.6 Outlier Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Global Sales — Box Plots (log₁⁺)', fontweight='bold')

for ax, (name, df), color in zip(axes, datasets.items(), COLORS):
    ax.boxplot(df[TARGET_COL].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(name, fontweight='bold')
    ax.set_ylabel('log₁⁺ Global Sales')
    ax.set_xticks([])

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'boxplots.png'), dpi=150)
plt.show()

### 6.7 EDA Summary

In [ ]:
print(f"{'Dataset':<12} {'Records':>10} {'Raw Skew':>10} "
      f"{'Mean Sales':>12} {'Median Sales':>14} {'Max Sales':>12}")
print("-" * 65)

for name, df in datasets.items():
    raw = df[TARGET_COL + '_raw']
    print(f"{name:<12} {len(df):>10,} {raw.skew():>10.2f} "
          f"{raw.mean():>11.2f}M {raw.median():>13.2f}M {raw.max():>11.2f}M")

## 7. Experimental Design <a name='design'></a>

### Mathematical Formulation

We model global video game sales as a regression problem. Given a feature vector **x** = [Platform, Genre, Publisher, NA_Sales, EU_Sales, JP_Sales, Other_Sales, Year], we seek a function *f* such that:

$$\hat{y} = f(\mathbf{x}) \approx \log_1(\text{Global\_Sales})$$

The objective is to minimise the Root Mean Squared Error (RMSE) over the test set:

$$Z = \text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

### Algorithms

| Algorithm | Justification |
|-----------|--------------|
| **Linear Regression** | Baseline model. Simple, interpretable, establishes a performance floor |
| **Decision Tree Regressor** | Captures non-linear interactions between features (e.g. genre × platform) |
| **Random Forest Regressor** | Ensemble method averaging many trees — reduces overfitting, improves robustness |

### Performance Metrics
- **RMSE** (primary) — average prediction error in log₁⁺ units. Lower is better.
- **R²** (secondary) — proportion of variance explained. Closer to 1.0 is better.

### Experiment Plan
Train all 3 algorithms on all 3 datasets. Use an 80/20 train-test split with 5-fold cross-validation on the training set. Record RMSE and R² for each of the 9 combinations.


## 8. Model Training & Evaluation <a name='models'></a>

In [ ]:
def get_X_y(df):
    feats = [c for c in FEATURE_COLS if c in df.columns]
    return df[feats].select_dtypes(include=[np.number]), df[TARGET_COL]

def run_model(model, X_train, X_test, y_train, y_test,
              model_name, dataset_name, scale=False):
    if scale:
        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

    model.fit(X_train, y_train)

    cv = cross_val_score(model, X_train, y_train,
                         cv=5, scoring='neg_root_mean_squared_error')

    y_pred = model.predict(X_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)

    print(f"    CV RMSE: {-cv.mean():.4f} ± {cv.std():.4f}  |  "
          f"Test RMSE: {rmse:.4f}  |  R²: {r2:.4f}")

    return {
        'Model': model_name, 'Dataset': dataset_name,
        'CV_RMSE': round(-cv.mean(), 4), 'CV_std': round(cv.std(), 4),
        'RMSE': round(rmse, 4), 'R2': round(r2, 4),
        'y_test': y_test.values, 'y_pred': y_pred,
        'model_obj': model,
        'feature_names': list(get_X_y(datasets[dataset_name])[0].columns),
    }

all_results = []

### 8.1 Linear Regression — Baseline

In [ ]:
print("LINEAR REGRESSION")
print("="*60)

for name, df in datasets.items():
    print(f"\n  {name}")
    X, y = get_X_y(df)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y,
                                               test_size=TEST_SIZE,
                                               random_state=RANDOM_STATE)
    result = run_model(LinearRegression(), X_tr, X_te, y_tr, y_te,
                       'Linear Regression', name, scale=True)
    all_results.append(result)

### 8.2 Decision Tree Regressor

In [ ]:
print("DECISION TREE")
print("="*60)

for name, df in datasets.items():
    print(f"\n  {name}")
    X, y = get_X_y(df)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y,
                                               test_size=TEST_SIZE,
                                               random_state=RANDOM_STATE)
    result = run_model(
        DecisionTreeRegressor(max_depth=10, min_samples_leaf=5,
                              random_state=RANDOM_STATE),
        X_tr, X_te, y_tr, y_te, 'Decision Tree', name
    )
    all_results.append(result)

### 8.3 Random Forest Regressor

In [ ]:
print("RANDOM FOREST")
print("="*60)

for name, df in datasets.items():
    print(f"\n  {name}")
    X, y = get_X_y(df)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y,
                                               test_size=TEST_SIZE,
                                               random_state=RANDOM_STATE)
    result = run_model(
        RandomForestRegressor(n_estimators=100, max_depth=15,
                              min_samples_leaf=3, n_jobs=-1,
                              random_state=RANDOM_STATE),
        X_tr, X_te, y_tr, y_te, 'Random Forest', name
    )
    all_results.append(result)

## 9. Sensitivity Analysis <a name='sensitivity'></a>

We examine how the Decision Tree's `max_depth` hyperparameter affects train vs test RMSE. This reveals the bias-variance tradeoff — shallow trees underfit, very deep trees overfit.


In [ ]:
df1 = datasets['Dataset 1']
X, y = get_X_y(df1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y,
                                           test_size=TEST_SIZE,
                                           random_state=RANDOM_STATE)

depths      = [2, 4, 6, 8, 10, 12, 15, 20, None]
train_rmses = []
test_rmses  = []

for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=RANDOM_STATE)
    m.fit(X_tr, y_tr)
    train_rmses.append(np.sqrt(mean_squared_error(y_tr, m.predict(X_tr))))
    test_rmses.append( np.sqrt(mean_squared_error(y_te, m.predict(X_te))))

labels = [str(d) if d else 'None' for d in depths]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(labels, train_rmses, marker='o', linewidth=2,
        label='Train RMSE', color=COLORS[0])
ax.plot(labels, test_rmses,  marker='s', linewidth=2,
        label='Test RMSE',  color=COLORS[1], linestyle='--')

chosen = labels.index('10')
ax.axvline(x=chosen, color='gray', linestyle=':', alpha=0.8)
ax.annotate('Chosen depth (10)',
            xy=(chosen, test_rmses[chosen]),
            xytext=(chosen + 0.5, test_rmses[chosen] + 0.02),
            fontsize=9, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_title('Sensitivity Analysis: Decision Tree max_depth vs RMSE (Dataset 1)',
             fontweight='bold')
ax.set_xlabel('max_depth')
ax.set_ylabel('RMSE (log₁⁺ scale)')
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'sensitivity_max_depth.png'), dpi=150)
plt.show()

print("Observation: train RMSE falls monotonically with depth (overfitting),")
print("while test RMSE bottoms out around depth 8–12 before rising again.")

We also examine how Random Forest's `n_estimators` affects performance — more trees generally help but with diminishing returns.

In [ ]:
estimators  = [10, 25, 50, 75, 100, 150, 200]
test_rmses_rf = []

for n in estimators:
    m = RandomForestRegressor(n_estimators=n, max_depth=15,
                              min_samples_leaf=3, n_jobs=-1,
                              random_state=RANDOM_STATE)
    m.fit(X_tr, y_tr)
    test_rmses_rf.append(np.sqrt(mean_squared_error(y_te, m.predict(X_te))))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(estimators, test_rmses_rf, marker='o', linewidth=2,
        color=COLORS[2], label='Test RMSE')
ax.axvline(x=100, color='gray', linestyle=':', alpha=0.8,
           label='Chosen (100 trees)')
ax.set_title('Sensitivity Analysis: Random Forest n_estimators vs RMSE (Dataset 1)',
             fontweight='bold')
ax.set_xlabel('Number of Trees (n_estimators)')
ax.set_ylabel('Test RMSE (log₁⁺)')
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'sensitivity_n_estimators.png'), dpi=150)
plt.show()

## 10. Results Summary <a name='results'></a>

In [ ]:
summary_df = pd.DataFrame([
    {'Model': r['Model'], 'Dataset': r['Dataset'],
     'CV RMSE': r['CV_RMSE'], 'CV Std': r['CV_std'],
     'Test RMSE': r['RMSE'], 'R²': r['R2']}
    for r in all_results
])

print("Full Results Table")
display(summary_df.to_string(index=False))

print("\nRMSE Pivot (lower is better):")
display(summary_df.pivot(index='Model', columns='Dataset', values='Test RMSE').round(4))

print("\nR² Pivot (higher is better):")
display(summary_df.pivot(index='Model', columns='Dataset', values='R²').round(4))

### 10.1 Model Comparison — RMSE and R²

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Across All Datasets',
             fontsize=14, fontweight='bold')

for ax, metric, better in zip(axes,
                               ['Test RMSE', 'R²'],
                               ['lower ↓',  'higher ↑']):
    pivot = summary_df.pivot(index='Dataset', columns='Model', values=metric)
    bars  = pivot.plot(kind='bar', ax=ax, color=COLORS,
                       edgecolor='white', width=0.7, alpha=0.9)
    ax.set_title(f'{metric}  ({better})', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Model', fontsize=9)
    ax.grid(True, axis='y', alpha=0.35)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.3f}',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'model_comparison.png'), dpi=150)
plt.show()

### 10.2 Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes      = axes.flatten()
model_colors = {
    'Linear Regression': COLORS[0],
    'Decision Tree':     COLORS[1],
    'Random Forest':     COLORS[2],
}

for i, res in enumerate(all_results):
    ax    = axes[i]
    color = model_colors[res['Model']]
    ax.scatter(res['y_test'], res['y_pred'],
               alpha=0.3, s=8, color=color)
    lims = [min(res['y_test'].min(), res['y_pred'].min()),
            max(res['y_test'].max(), res['y_pred'].max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_title(f"{res['Model']}\n{res['Dataset']}",
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Actual (log₁⁺)')
    ax.set_ylabel('Predicted')
    ax.text(0.04, 0.93,
            f"RMSE {res['RMSE']}\nR² {res['R2']}",
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round', fc='white', alpha=0.85))
    ax.legend(fontsize=7)

fig.suptitle('Actual vs Predicted Global Sales (log₁⁺)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'actual_vs_predicted.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### 10.3 Feature Importance

In [ ]:
tree_results = [r for r in all_results
                if r['Model'] in ('Decision Tree', 'Random Forest')]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, res in enumerate(tree_results):
    ax   = axes[i]
    imp  = res['model_obj'].feature_importances_
    feat = res['feature_names']
    df_i = (pd.DataFrame({'Feature': feat, 'Importance': imp})
              .sort_values('Importance', ascending=True))
    color = COLORS[1] if res['Model'] == 'Decision Tree' else COLORS[2]
    df_i.plot(kind='barh', x='Feature', y='Importance',
              ax=ax, legend=False, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f"{res['Model']}\n{res['Dataset']}", fontweight='bold')
    ax.set_xlabel('Importance')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Importance — Tree-Based Models',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'feature_importance.png'), dpi=150)
plt.show()

## 11. Discussion & Conclusion <a name='discussion'></a>

> **Complete this section after running all experiments.** The guiding questions below map directly to the rubric's Discussion criterion.

### 11.1 Which model performed best?
*(Fill in after results are generated)*  
Based on the RMSE and R² results across all three datasets, [MODEL] consistently achieved the lowest test RMSE, suggesting it generalises best to unseen data. [BRIEF REASON].

### 11.2 Why did the best model outperform the others?
*(Fill in)*  
Random Forest's advantage over a single Decision Tree stems from variance reduction through averaging — individual trees overfit to different noise patterns, but their average cancels much of this out. Linear Regression struggled because...

### 11.3 Were results consistent across datasets?
*(Fill in)*  
Results were [consistent / inconsistent] across the three datasets. Dataset [X] showed markedly [higher/lower] RMSE, which may be explained by...

### 11.4 What do the feature importance plots tell us?
*(Fill in)*  
The most predictive features were [X, Y, Z]. Regional sales features (NA_Sales, EU_Sales) dominated importance scores, which makes intuitive sense — a game that already sells well in one region is likely a known title with established demand elsewhere.

### 11.5 Limitations
- The datasets are predominantly pre-2017 and do not capture the modern digital distribution era (Steam, Epic Games Store, mobile)
- Review scores, marketing spend, and franchise recognition are absent but likely strong predictors
- Label-encoding of categorical features (Genre, Publisher, Platform) introduces an artificial ordinal relationship that tree models partially compensate for

### 11.6 Future Work
- Include review score data (Metacritic) as an additional feature
- Apply one-hot encoding for categoricals and compare against label encoding
- Explore gradient boosting methods (XGBoost, LightGBM) as stronger ensemble alternatives
- Extend dataset to post-2017 digital sales figures

### 11.7 Stakeholder Takeaway
*(Write 2–3 plain-English sentences a publisher could understand)*  
Our best model can predict global sales within approximately [X]M units on average. The most reliable predictors of sales are regional performance figures and platform — suggesting publishers should prioritise early regional launch data when forecasting global performance. Independent developers can use this model to benchmark how genre and platform choices affect expected sales before committing to full production.


In [ ]:
# Final summary — best model per dataset
print("Best model per dataset (lowest test RMSE):\n")
rmse_pivot = summary_df.pivot(index='Model', columns='Dataset', values='Test RMSE')

for col in rmse_pivot.columns:
    best = rmse_pivot[col].idxmin()
    val  = rmse_pivot[col].min()
    r2   = summary_df[(summary_df['Model'] == best) &
                      (summary_df['Dataset'] == col)]['R²'].values[0]
    print(f"  {col}:  {best}  (RMSE = {val:.4f}, R² = {r2:.4f})")

print("\nAll plots saved to:", PLOT_DIR)
for f in sorted(os.listdir(PLOT_DIR)):
    print(f"  {f}")